# Comparacion de cobertura nutricional para poblacion guatemalteca 

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

HISTORICOS = PROJECT_ROOT / 'data' / 'raw' / 'ine' / 'cba_historicos'

CBA_RURAL_PATH = HISTORICOS / 'CBAR-2024-2026.xlsx'
CBA_URBANA_PATH = HISTORICOS / 'CBAU-2024-2026.xlsx'


In [2]:
from canasta_inteligente.application.cba_pipeline import load_catalog_pickle

catalog_path = PROJECT_ROOT / 'data' / 'processed' / 'cba_catalog.pkl'
if not catalog_path.exists():
    raise FileNotFoundError(
        f'No existe {catalog_path}. Ejecuta primero cba_pipeline.ipynb '
        'o el módulo canasta_inteligente.application.cba_pipeline.'
    )
catalog = load_catalog_pickle(catalog_path)

In [3]:
from canasta_inteligente.nutrition.evaluation._implementation import evaluacion_de_requerimientos_diarios, Peso
from canasta_inteligente.domain.persona import Persona
from statistics import mean
from canasta_inteligente.application.demo_canasta import seleccionar_catalogo_region

"""
https://www.prensalibre.com/guatemala/comunitario/por-que-los-guatemaltecos-son-los-mas-bajos-de-estatura-del-mundo/

Hombre: 19 años - 1.64m 
Mujer: 19 años - 1.49m

"""
altura_mujer_ej = 1.49
altura_hombre_ej = 1.64
mujer_ej = Persona(nombre="Mujer de referencia", edad=19, sexo='mujer', altura=altura_mujer_ej, peso=Peso.obetener_peso_de_referencia(altura_mujer_ej), naf='low')
hombre_ej = Persona(nombre="Hombre de referencia", edad=19, sexo='hombre', altura=altura_hombre_ej,peso=Peso.obetener_peso_de_referencia(altura_hombre_ej), naf='low')

req_mujer_promedio = evaluacion_de_requerimientos_diarios(mujer_ej)
req_hombre_promedio = evaluacion_de_requerimientos_diarios(hombre_ej)

ree_mujer = req_mujer_promedio['energia']['ree']
ree_hombre = req_hombre_promedio['energia']['ree']

print('====REE====')
print(f'Mujer: {ree_mujer}')
print(f'Hombre: {ree_hombre}')
print(f'Promedio: {mean([ree_mujer, ree_hombre])}')


====REE====
Mujer: 1876.03276538
Hombre: 2453.86817552
Promedio: 2164.95047045


# Comparacion rural - Agosto 2026

In [4]:
catalog.get('arroz')

Food(id='arroz', name='ARROZ', category='CEREALES', aliases={'Arroz corriente', 'ARROZ', 'Arroz'}, price_timelines={'general': <canasta_inteligente.domain.prices.PriceTimeline object at 0x0000019AA7438050>, 'rural': <canasta_inteligente.domain.prices.PriceTimeline object at 0x0000019AA7406350>, 'urbana': <canasta_inteligente.domain.prices.PriceTimeline object at 0x0000019AA7406490>}, nutrition=NutritionProfile(incap_code='13004', incap_name='ARROZ BLANCO, GRANO MEDIANO, CRUDO, S/ENRIQ.', category='CEREALES, GRANOS SECOS Y DERIVADOS', values_per_100g={'energia_kcal': 360.0, 'agua_pct': 13.0, 'fraccion_comestible_pct': 1.0, 'proteina_g': 6.61, 'grasa_total_g': 0.58, 'ag_sat_g': 0.16, 'ag_mono_g': 0.18, 'ag_poli_g': 0.16, 'colesterol_mg': 0.0, 'carbohidratos_g': 79.34, 'azucares_g': 0.12, 'fibra_dietetica_g': 1.3, 'ceniza_g': 0.58, 'calcio_mg': 9.0, 'hierro_mg': 0.8, 'magnesio_mg': 35.0, 'fosforo_mg': 108.0, 'potasio_mg': 86.0, 'sodio_mg': 1.0, 'zinc_mg': 1.16, 'cobre_mg': 0.11, 'selenio_

In [5]:
from canasta_inteligente.application.demo_canasta import seleccionar_catalogo_region
import unicodedata


dias = 31
catalog_rural = seleccionar_catalogo_region(catalog, 'rural')

cba_rural = pd.read_excel(CBA_RURAL_PATH, sheet_name='Histórico producto CBAR')
cba_rural_last_canasta = cba_rural[(cba_rural['Año'] == 2026) & (cba_rural['Mes'] == 'Agosto')]
cba_rural_last_canasta['slug'] = cba_rural_last_canasta['Producto'].apply(lambda product_name: "".join(char for char in unicodedata.normalize("NFD", str.lower(product_name).replace(" ", "_")) if unicodedata.category(char) != "Mn") )
print(cba_rural_last_canasta['slug'])


1860                                      arroz_corriente
1861                                      arroz_precocido
1862                                          maiz_blanco
1863                                       harina_de_maiz
1864              harina_para_atoles_(incluye_incaparina)
1865                                          pan_frances
1866                                            pan_dulce
1867                                      galletas_dulces
1868                                    tortillas_frescas
1869                                           avena/mosh
1870                                            espagueti
1871    fideos_en_todas_sus_formas_(excepto_macarrones...
1872                               frijoles_negros,_secos
1873    frijoles_preparados,_procesados_y_condimentado...
1874                        repollo,_fresco_o_refrigerado
1875                      cilantro,_perejil_y_hierbabuena
1876                            macuy/hierba_mora/quilete
1877          

In [ ]:
food_id = catalog_rural.resolve_by_alias('arroz_corriente')

In [ ]:
cba_rural_last_canasta['Kilocalorías diarias'].sum()

np.float64(2172.000000000001)

In [ ]:
costo_mensual_real = cba_rural_last_canasta['Costo mensual'].sum()
costo_diario_real = cba_rural_last_canasta['Costo diario'].sum()
costo_diario_real

In [ ]:
req = hombre_ej.calcular_requerimientos()
resultados_hombre_ej = hombre_ej.calculo_canasta_diaria_costo_minimizado(catalog)
resultado_min = float('inf')
resultado_optimo = None
for resultado in resultados_hombre_ej:
    if resultado['costo_total_q'] < resultado_min:
        resultado_min = resultado['costo_total_q']
        resultado_optimo = resultado

resultado_optimo

{'escenario': 'hierro_baja_disponiblidad',
 'estado': 'Optimal',
 'codigo_estado': 1,
 'costo_total_q': 24.372305398526827,
 'valor_objetivo': 24.372305398526823,
 'desviaciones': {'energia_deficit_kcal': 0.0,
  'energia_exceso_kcal': np.float64(8.068000170169398e-07),
  'colesterol_exceso_mg': np.float64(1.1904000416507188e-06)},
 'penalizaciones': {'energia': 1.0, 'colesterol': 0.1},
 'alimentos':                                alimento_id  \
 0                                    arroz   
 1                      avena_de_toda_clase   
 2                              pan_frances   
 3                                pan_dulce   
 4                                   fideos   
 ..                                     ...   
 67  carne_de_res_para_asar_con_y_sin_hueso   
 68         jamon_pollo_res_cerdo_mixto_etc   
 69                jugos_de_frutas_en_polvo   
 70                                mayonesa   
 71                                  atoles   
 
                                

# Exploración inicial de ENIGH

Este notebook explora las bases de **personas** y **hogares** de ENIGH y utiliza los metadatos de los archivos `.sav` para interpretar las variables codificadas.

El objetivo inicial es entender la estructura de los datos antes de construir el generador de hogares.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ine' / 'enigh'

PERSONAS_XLSX = DATA_DIR / "enigh_personas.xlsx"
HOGARES_XLSX = DATA_DIR / "enigh_hogares.xlsx"

PERSONAS_SAV = DATA_DIR / "ENIGH_Personas_20232710.sav"
HOGARES_SAV = DATA_DIR / "ENIGH_Hogares_20232710.sav"

## 1. Carga de las bases

In [ ]:
personas = pd.read_excel(PERSONAS_XLSX)
hogares = pd.read_excel(HOGARES_XLSX)

print("Personas:", personas.shape)
print("Hogares :", hogares.shape)

KeyboardInterrupt: 

In [ ]:
display(personas.head())
display(hogares.head())

,DEPTO,MUPIO,AREA,HOGAR,PONDERADOR,ID,B1P00A02,B1P00A03,B1P00A05,B1P00A06,...,B1P03D12A,B1P03D12B,B1P03D13A,B1P03D13B,B1P03D14A,B1P03D14B,B1P03E01A,B1P03E01B,B1P03E02A,B1P03E02B
0,1,101,1,1,900,3,1,17,3,5.0,...,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN
1,1,101,1,1,900,4,2,14,3,5.0,...,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN
2,1,101,1,1,900,5,2,10,3,5.0,...,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN
3,1,101,1,1,900,2,2,45,2,5.0,...,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN
4,1,101,1,1,900,1,1,48,1,5.0,...,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN


,DEPTO,MUPIO,AREA,HOGAR,PONDERADOR,B1PPB01,B1PPB02,B1PPB03,B1PPB04,B1P00C07,...,B1P01E09G,B1P01E09H,B1P01E09I,B1P01E09J,B1P01E010,B1P01E011,B1P01E012A,B1P01E012B,B1P01E013A,B1P01E014A
0,11,1106,1,6502,213,2,1,1,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11,1106,1,6503,213,2,1,1,5,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11,1106,1,6501,213,2,1,1,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11,1106,1,6500,213,2,1,1,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11,1106,1,6499,213,2,1,1,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
!pip install pyreadstat


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pyreadstat

_, meta_personas = pyreadstat.read_sav(PERSONAS_SAV, metadataonly=True)
_, meta_hogares = pyreadstat.read_sav(HOGARES_SAV, metadataonly=True)

print("Variables personas:", len(meta_personas.column_names))
print("Variables hogares :", len(meta_hogares.column_names))

Variables personas: 234
Variables hogares : 157


In [ ]:
def construir_diccionario(meta):
    labels = dict(zip(meta.column_names, meta.column_labels))
    value_labels = meta.variable_value_labels

    rows = []
    for variable in meta.column_names:
        rows.append({
            "variable": variable,
            "descripcion": labels.get(variable),
            "valores": value_labels.get(variable)
        })

    return pd.DataFrame(rows)

dic_personas = construir_diccionario(meta_personas)
dic_hogares = construir_diccionario(meta_hogares)

display(dic_personas.loc[dic_personas['valores'].isna()].head(20))

,variable,descripcion,valores
3,HOGAR,Número de hogar,None
4,PONDERADOR,Factor de ponderación,None
5,ID,Código de la persona en la boleta,None
7,B1P00A03,Edad,None
31,B1P00C03,Promedio a la semana,None
33,B1P00C04B,Promedio a la semana,None
56,B1P02B06A,Tiempo laborando años,None
57,B1P02B06B,Tiempo laborando meses,None
59,B1P02B08,Sueldo o salario mensual,None
61,B1P02B09B,Quetzales horas extras,None


In [ ]:
def buscar_variables(diccionario, texto):
    mask = (
        diccionario["variable"].astype(str).str.contains(texto, case=False, na=False)
        | diccionario["descripcion"].astype(str).str.contains(texto, case=False, na=False)
    )
    return diccionario.loc[mask]

buscar_variables(dic_personas, "edad")

,variable,descripcion,valores
7,B1P00A03,Edad,None


In [ ]:
buscar_variables(dic_personas, "sexo")

,variable,descripcion,valores
6,B1P00A02,Sexo,"{1.0: 'Hombre', 2.0: 'Mujer'}"


In [ ]:
buscar_variables(dic_personas, "parentesco")

,variable,descripcion,valores
8,B1P00A05,Parentesco,"{1.0: 'Jefe (a) del hogar', 2.0: 'Esposo (a) o..."


In [ ]:
identificacion = ["DEPTO", "MUPIO", "AREA", "HOGAR", "PONDERADOR"]

display(personas[identificacion + ["ID"]].head(15))
display(hogares[identificacion].head())

,DEPTO,MUPIO,AREA,HOGAR,PONDERADOR,ID
0,1,101,1,1,900,3
1,1,101,1,1,900,4
2,1,101,1,1,900,5
3,1,101,1,1,900,2
4,1,101,1,1,900,1
5,1,101,1,2,900,1
6,1,101,1,2,900,2
7,1,101,1,3,900,2
8,1,101,1,3,900,1
9,1,101,1,4,900,1


,DEPTO,MUPIO,AREA,HOGAR,PONDERADOR
0,11,1106,1,6502,213
1,11,1106,1,6503,213
2,11,1106,1,6501,213
3,11,1106,1,6500,213
4,11,1106,1,6499,213


In [ ]:
for col in identificacion:
    print(
        col,
        "| personas:", personas[col].nunique(dropna=False),
        "| hogares:", hogares[col].nunique(dropna=False)
    )

DEPTO | personas: 22 | hogares: 22
MUPIO | personas: 341 | hogares: 341
AREA | personas: 2 | hogares: 2
HOGAR | personas: 13790 | hogares: 13790
PONDERADOR | personas: 407 | hogares: 407


In [ ]:
hogar_cols = ["DEPTO", "MUPIO", "AREA", "HOGAR"]

personas["hogar_id"] = (
    personas[hogar_cols]
    .astype(str)
    .agg("-".join, axis=1)
)

hogares["hogar_id"] = (
    hogares[hogar_cols]
    .astype(str)
    .agg("-".join, axis=1)
)

print("Hogares distintos en personas:", personas["hogar_id"].nunique())
print("Filas de hogares:", len(hogares))
print("hogar_id únicos en hogares:", hogares["hogar_id"].is_unique)

C:\Users\lp109\AppData\Local\Temp\ipykernel_8640\1417378212.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  personas["hogar_id"] = (


Hogares distintos en personas: 13790
Filas de hogares: 13790
hogar_id únicos en hogares: True


C:\Users\lp109\AppData\Local\Temp\ipykernel_8640\1417378212.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hogares["hogar_id"] = (


## 6. Tamaño de los hogares

In [ ]:
tamano_hogar = (
    personas.groupby("hogar_id")
    .size()
    .rename("personas")
)

display(tamano_hogar.describe())

count    13790.000000
mean         4.440174
std          2.240752
min          1.000000
25%          3.000000
50%          4.000000
75%          6.000000
max         19.000000
Name: personas, dtype: float64

In [ ]:
tamano_hogar.value_counts().sort_index().to_frame("cantidad_hogares")

,cantidad_hogares
personas,
1,792
2,1762
3,2493
4,2878
5,2330
6,1419
7,860
8,509
9,324


In [ ]:
hogar_ejemplo = personas["hogar_id"].iloc[0]

columnas_base = [
    "hogar_id", "ID", "DEPTO", "MUPIO", "AREA",
    "HOGAR", "PONDERADOR"
]

display(
    personas.loc[
        personas["hogar_id"] == hogar_ejemplo,
        columnas_base + ["B1P00A02", "B1P00A03", "B1P00A05"]
    ]
)

,hogar_id,ID,DEPTO,MUPIO,AREA,HOGAR,PONDERADOR,B1P00A02,B1P00A03,B1P00A05
0,1-101-1-1,3,1,101,1,1,900,1,17,3
1,1-101-1-1,4,1,101,1,1,900,2,14,3
2,1-101-1-1,5,1,101,1,1,900,2,10,3
3,1-101-1-1,2,1,101,1,1,900,2,45,2
4,1-101-1-1,1,1,101,1,1,900,1,48,1


In [ ]:
def describir_variable(df, diccionario, variable):
    info = diccionario.loc[diccionario["variable"] == variable]

    if info.empty:
        print("Variable no encontrada:", variable)
        return

    display(info)

    print("\nFrecuencias:")
    display(
        df[variable]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("n")
    )

describir_variable(personas, dic_personas, "B1P00A03")

,variable,descripcion,valores
7,B1P00A03,Edad,None



Frecuencias:


,n
B1P00A03,
0,1298
1,1127
2,1241
3,1341
4,1430
...,...
97,7
98,3
99,4


In [ ]:
describir_variable(personas, dic_personas, "B1P00A02")

,variable,descripcion,valores
6,B1P00A02,Sexo,"{1.0: 'Hombre', 2.0: 'Mujer'}"



Frecuencias:


,n
B1P00A02,
1,29085
2,32145


In [ ]:
describir_variable(personas, dic_personas, "B1P00A05")

,variable,descripcion,valores
8,B1P00A05,Parentesco,"{1.0: 'Jefe (a) del hogar', 2.0: 'Esposo (a) o..."



Frecuencias:


,n
B1P00A05,
1,13790
2,9601
3,27645
4,536
5,1646
6,5161
7,668
8,265
9,573


## 9. Cobertura y valores faltantes

Primero se revisan las variables con mayor y menor disponibilidad antes de seleccionar las que usará el generador.

In [ ]:
resumen_personas = pd.DataFrame({
    "tipo": personas.dtypes.astype(str),
    "no_nulos": personas.notna().sum(),
    "faltantes": personas.isna().sum(),
    "pct_faltantes": personas.isna().mean().mul(100).round(2),
    "unicos": personas.nunique(dropna=True)
})

display(resumen_personas.sort_values("pct_faltantes").head(25))

,tipo,no_nulos,faltantes,pct_faltantes,unicos
DEPTO,int64,61230,0,0.00,22
MUPIO,int64,61230,0,0.00,341
AREA,int64,61230,0,0.00,2
HOGAR,int64,61230,0,0.00,13790
PONDERADOR,int64,61230,0,0.00,407
ID,int64,61230,0,0.00,19
B1P00A02,int64,61230,0,0.00,2
B1P00A03,int64,61230,0,0.00,102
B1P00A05,int64,61230,0,0.00,14
hogar_id,str,61230,0,0.00,13790


In [ ]:
display(
    resumen_personas
    .sort_values("pct_faltantes", ascending=False)
    .head(25)
)

,tipo,no_nulos,faltantes,pct_faltantes,unicos
B1P03D14B,float64,0,61230,100.00,0
B1P03A04B,float64,1,61229,100.00,1
B1P03A09B,float64,2,61228,100.00,2
B1P03A05B,float64,2,61228,100.00,2
B1P02C22B,float64,3,61227,100.00,3
B1P02C14B,float64,3,61227,100.00,3
B1P02C09B,float64,3,61227,100.00,3
B1P03B08B,float64,1,61229,100.00,1
B1P02C17B,float64,9,61221,99.99,7
B1P03B09B,float64,6,61224,99.99,4


## 10. Tabla compacta del diccionario

Se combinan metadatos y estadísticas observadas para facilitar la revisión.

In [ ]:
def resumen_diccionario(df, diccionario):
    estadisticas = pd.DataFrame({
        "variable": df.columns,
        "tipo": [str(df[c].dtype) for c in df.columns],
        "no_nulos": [df[c].notna().sum() for c in df.columns],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "pct_faltantes": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    })

    return diccionario.merge(estadisticas, on="variable", how="right")

catalogo_personas = resumen_diccionario(personas, dic_personas)
catalogo_hogares = resumen_diccionario(hogares, dic_hogares)

display(catalogo_personas.head(30))

,variable,descripcion,valores,tipo,no_nulos,unicos,pct_faltantes
0,DEPTO,Departamento,"{1.0: 'Guatemala', 2.0: 'El Progreso', 3.0: 'S...",int64,61230,22,0.00
1,MUPIO,Municipio,"{101.0: 'Zona 1', 102.0: 'Zona 2', 103.0: 'Zon...",int64,61230,341,0.00
2,AREA,Área,"{1.0: 'URBANO', 2.0: 'RURAL'}",int64,61230,2,0.00
3,HOGAR,Número de hogar,None,int64,61230,13790,0.00
4,PONDERADOR,Factor de ponderación,None,int64,61230,407,0.00
5,ID,Código de la persona en la boleta,None,int64,61230,19,0.00
6,B1P00A02,Sexo,"{1.0: 'Hombre', 2.0: 'Mujer'}",int64,61230,2,0.00
7,B1P00A03,Edad,None,int64,61230,102,0.00
8,B1P00A05,Parentesco,"{1.0: 'Jefe (a) del hogar', 2.0: 'Esposo (a) o...",int64,61230,14,0.00
9,B1P00A06,Código de pueblo indígena,"{1.0: 'Maya', 2.0: 'Garifuna', 3.0: 'Xinka', 4...",float64,61196,6,0.06


## 11. Variables candidatas para el generador

La selección final debe hacerse usando las etiquetas del `.sav`, no por inferencia basada únicamente en los códigos.

In [ ]:
terminos = [
    "edad",
    "sexo",
    "parentesco",
    "embar",
    "lact",
    "peso",
    "talla",
    "altura"
]

candidatas = pd.concat(
    [buscar_variables(dic_personas, termino) for termino in terminos],
    ignore_index=True
).drop_duplicates("variable")

display(candidatas)

,variable,descripcion,valores
0,B1P00A03,Edad,None
1,B1P00A02,Sexo,"{1.0: 'Hombre', 2.0: 'Mujer'}"
2,B1P00A05,Parentesco,"{1.0: 'Jefe (a) del hogar', 2.0: 'Esposo (a) o..."


## 12. Relación personas-hogares

Se verifica cuántos registros de personas encuentran su hogar correspondiente.

In [ ]:
merge_test = personas[["hogar_id"]].merge(
    hogares[["hogar_id"]],
    on="hogar_id",
    how="left",
    indicator=True
)

merge_test["_merge"].value_counts()

_merge
both          61230
left_only         0
right_only        0
Name: count, dtype: int64

In [ ]:
from canasta_inteligente.nutrition.evaluation._implementation import evaluacion_de_requerimientos_diarios, Peso
from canasta_inteligente.domain.persona import Persona
Peso.obetener_peso_de_referencia(1.49)

48.8422